# HanuAi ML Assessment: Task 1, Web Scraping and Sentiment Analysis

**What this notebook does:** (1) collects public customer reviews for one product on bestbuy.ca, (2) cleans them into a table with the required fields, (3) labels each review Positive, Negative or Neutral, (4) tags each review with the product aspects it talks about, and (5) summarises what drives satisfaction and dissatisfaction.

**Important, read before running**

* **Pick one product on bestbuy.ca with at least 50 reviews** and paste its address into `PRODUCT_URL` in section 2.
* **This notebook was written without access to the live site.** The scraper uses text-based locators (button names, labels) and structured data on the page, which are the least fragile options, but the page may differ. Section 4 has an *inspect* cell that tells you in seconds what the page contains, so you can adjust the `SELECTORS` in section 2 if needed. Run the self-test in section 3 first: it checks the parsing and sentiment code on invented text, not on real data.
* **Be a polite visitor.** Only public review text is collected, requests are slowed down, and the scraper stops (it does not try to bypass) if the site shows a CAPTCHA or an access-denied page. Check the site's terms and `robots.txt`; section 2 prints what `robots.txt` says.
* The code cells contain no hash comments. Each step is explained in the text cells and in function docstrings.

**Install once:** `pip install playwright beautifulsoup4 pandas vaderSentiment matplotlib openpyxl` then `playwright install chromium`.

## 1. Imports

In [1]:
import hashlib
import json
import os
import random
import re
import time
import urllib.robotparser
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

## 2. Configuration
Everything you may need to change is in this one cell.

* `SORT_LABELS` are the review sort options to loop through, so that each sort order can reveal reviews the others do not. The names must match the labels shown on the page ("Most Helpful", "Newest", "Highest Rating", "Lowest Rating", "Most Relevant"). If a label is not found it is skipped and reported.
* `SELECTORS` are optional CSS selectors for review cards. Leave them empty to rely on the structured data embedded in the page; fill them in after using the inspect cell if needed.
* `PROXY_LIST` is empty by default. See the report for when proxies are appropriate.

In [2]:
PRODUCT_URL = "PASTE_A_BESTBUY_CA_PRODUCT_URL_HERE"
SOURCE_NAME = "BestBuy Canada"
SORT_LABELS = ["Most Helpful", "Newest", "Highest Rating", "Lowest Rating", "Most Relevant"]
HEADLESS = True
MAX_SHOW_MORE_CLICKS = 150
MIN_DELAY_S = 1.5
MAX_DELAY_S = 4.0
MAX_RETRIES = 4
PROXY_LIST = []

SELECTORS = {
    "card": "",
    "title": "",
    "body": "",
    "author": "",
    "date": "",
    "rating": "",
}

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
]
BLOCK_WORDS = r"captcha|verify you are (a )?human|access denied|unusual traffic|are you a robot|request blocked"

In [3]:
def check_robots(url, agent="*"):
    """Print what the site's robots.txt says about this URL, so the choice to scrape is an informed one."""
    root = re.match(r"https?://[^/]+", url)
    if not root:
        print("set PRODUCT_URL first")
        return None
    parser = urllib.robotparser.RobotFileParser()
    parser.set_url(root.group(0) + "/robots.txt")
    try:
        parser.read()
    except Exception as error:
        print("could not read robots.txt:", error)
        return None
    allowed = parser.can_fetch(agent, url)
    print("robots.txt allows this URL for generic crawlers:", allowed)
    return allowed


if PRODUCT_URL.startswith("http"):
    check_robots(PRODUCT_URL)
else:
    print("Paste your product URL into PRODUCT_URL above.")

Paste your product URL into PRODUCT_URL above.


## 3. Helper functions (parsing, dates, keys) and a self-test
Reviews can be found in two ways, tried in this order:

1. **Page cards:** if you provided `SELECTORS["card"]`, each card is parsed with those selectors.
2. **Structured data:** many shops embed their reviews as `application/ld+json` (schema.org `Review`). This is stable across page redesigns, so it is the default.

Every review gets a **primary key**: a short hash of reviewer, title, date and the first words of the text. The same review seen under two different sort orders gets the same key, so duplicates collapse automatically.

In [4]:
def clean_space(value):
    """Collapse all whitespace (including non-breaking spaces) into single spaces and trim the ends."""
    return re.sub(r"\s+", " ", str(value or "").replace("\xa0", " ")).strip()


def parse_review_date(text, today=None):
    """Turn a date such as 'March 5, 2024', '2024-03-05' or '3 weeks ago' into YYYY-MM-DD, or None if it cannot be read.

    Relative dates are converted using today's date, so they are approximate (a month is counted as 30 days).
    """
    today = today or date.today()
    text = clean_space(text).lower()
    if not text:
        return None
    if "today" in text:
        return today.isoformat()
    if "yesterday" in text:
        return (today - timedelta(days=1)).isoformat()
    relative = re.search(r"(\d+)\s*(day|week|month|year)s?\s+ago", text)
    if relative:
        days = {"day": 1, "week": 7, "month": 30, "year": 365}[relative.group(2)]
        return (today - timedelta(days=int(relative.group(1)) * days)).isoformat()
    try:
        return date_parser.parse(text, fuzzy=True).date().isoformat()
    except (ValueError, OverflowError):
        return None


def parse_rating(value):
    """Read a star rating out of text such as '4', '4.0', 'Rated 4 out of 5 stars' or '4/5'. Returns a float from 1 to 5, or None."""
    match = re.search(r"(\d(?:\.\d)?)\s*(?:out of|/)\s*5", str(value or ""))
    if not match:
        match = re.fullmatch(r"\s*(\d(?:\.\d)?)\s*", str(value or ""))
    if not match:
        return None
    number = float(match.group(1))
    return number if 1 <= number <= 5 else None


def review_key(review):
    """Build a stable primary key from the reviewer, title, date and the start of the review text."""
    raw = "|".join(clean_space(review.get(k, "")).lower() for k in ("author", "title", "date")) + "|" + clean_space(review.get("body", "")).lower()[:80]
    return "R" + hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12].upper()


def walk_json(node):
    """Yield every dictionary found anywhere inside nested lists and dictionaries."""
    if isinstance(node, dict):
        yield node
        for value in node.values():
            yield from walk_json(value)
    elif isinstance(node, list):
        for value in node:
            yield from walk_json(value)


def reviews_from_jsonld(html):
    """Read schema.org Review objects from the page's embedded JSON-LD blocks."""
    soup = BeautifulSoup(html, "html.parser")
    found = []
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or tag.get_text() or "")
        except json.JSONDecodeError:
            continue
        for node in walk_json(data):
            kind = node.get("@type")
            if kind == "Review" or (isinstance(kind, list) and "Review" in kind):
                author = node.get("author")
                author = author.get("name") if isinstance(author, dict) else author
                rating = node.get("reviewRating") or {}
                found.append({
                    "author": clean_space(author),
                    "title": clean_space(node.get("name")),
                    "body": clean_space(node.get("reviewBody") or node.get("description")),
                    "date": parse_review_date(node.get("datePublished")),
                    "rating": parse_rating(rating.get("ratingValue") if isinstance(rating, dict) else rating),
                })
    return found


def reviews_from_cards(html, selectors):
    """Read reviews from page cards using the CSS selectors given in SELECTORS."""
    soup = BeautifulSoup(html, "html.parser")
    found = []

    def pick(card, css):
        element = card.select_one(css) if css else None
        if element is None:
            return ""
        return clean_space(element.get("aria-label") or element.get("content") or element.get_text(" "))

    for card in soup.select(selectors["card"]):
        rating_text = pick(card, selectors["rating"]) or card.get_text(" ")
        found.append({
            "author": pick(card, selectors["author"]),
            "title": pick(card, selectors["title"]),
            "body": pick(card, selectors["body"]),
            "date": parse_review_date(pick(card, selectors["date"])),
            "rating": parse_rating(rating_text),
        })
    return found


def extract_reviews(html, selectors=None):
    """Return the reviews found in a page: card selectors first if provided, structured data otherwise."""
    selectors = selectors or SELECTORS
    if selectors.get("card"):
        cards = reviews_from_cards(html, selectors)
        if cards:
            return cards
    return reviews_from_jsonld(html)

In [5]:
SAMPLE_HTML = """
<html><body><script type="application/ld+json">
{"@context": "https://schema.org", "@type": "Product", "name": "Test Widget",
 "review": [
  {"@type": "Review", "author": {"@type": "Person", "name": "Sam"}, "name": "Works great",
   "reviewBody": "Easy to set up and the quality is excellent.", "datePublished": "2024-03-05",
   "reviewRating": {"@type": "Rating", "ratingValue": "5"}},
  {"@type": "Review", "author": "Priya", "name": "Stopped working",
   "reviewBody": "Broke after two weeks. Very disappointed.", "datePublished": "March 9, 2024",
   "reviewRating": {"ratingValue": 1}}
 ]}
</script></body></html>
"""

parsed = extract_reviews(SAMPLE_HTML, {"card": ""})
assert len(parsed) == 2 and parsed[0]["date"] == "2024-03-05" and parsed[1]["date"] == "2024-03-09" and parsed[1]["rating"] == 1.0
assert parse_review_date("3 weeks ago", today=date(2024, 6, 22)) == "2024-06-01"
assert parse_rating("Rated 4 out of 5 stars") == 4.0 and parse_rating("9") is None
assert review_key(parsed[0]) == review_key(dict(parsed[0])) and review_key(parsed[0]) != review_key(parsed[1])
print("parsing self-test passed on invented sample HTML (this is test data, not part of the dataset)")

parsing self-test passed on invented sample HTML (this is test data, not part of the dataset)


## 4. The scraper
**Flow:** open the product page, dismiss pop-ups, scroll to the reviews, then for each sort option: choose the sort, click *Show more* until nothing new loads, and read all reviews on the page. Reviews seen under several sort orders are merged, and we record which sort orders found each one.

**Safeguards built in** (the report explains why each one exists):

* random pauses between actions, so requests are spread out like a human reader's;
* a realistic browser identity and Canadian English locale;
* retries with exponential backoff and random jitter for temporary failures;
* an optional proxy list, rotated per run;
* a stop on CAPTCHA or access-denied pages. The scraper **does not try to solve or bypass them**; it saves what it has and tells you.

In [6]:
class BlockedError(RuntimeError):
    """Raised when the site shows a CAPTCHA or an access-denied page."""


def polite_sleep(low=None, high=None):
    """Wait a random time between low and high seconds."""
    time.sleep(random.uniform(low or MIN_DELAY_S, high or MAX_DELAY_S))


def with_retries(action, label, retries=None):
    """Run action(); on a temporary error wait 2, 4, 8... seconds plus random jitter and try again, up to the retry limit."""
    retries = MAX_RETRIES if retries is None else retries
    for attempt in range(retries + 1):
        try:
            return action()
        except BlockedError:
            raise
        except Exception as error:
            if attempt == retries:
                raise
            wait = (2 ** (attempt + 1)) + random.uniform(0, 1.5)
            print(f"{label} failed ({type(error).__name__}), retry {attempt + 1}/{retries} in {wait:.1f}s")
            time.sleep(wait)


def check_not_blocked(page):
    """Raise BlockedError if the visible page text looks like a CAPTCHA or access-denied page."""
    text = page.inner_text("body")[:3000].lower()
    if re.search(BLOCK_WORDS, text):
        raise BlockedError("The site is asking for human verification or denied access. Stop, wait, or solve it by hand in a visible browser (HEADLESS = False).")


def dismiss_popups(page):
    """Close a cookie or location banner if one is present. Missing buttons are ignored."""
    for name in (r"accept", r"got it", r"close", r"no thanks", r"continue"):
        try:
            button = page.get_by_role("button", name=re.compile(name, re.I)).first
            if button.is_visible(timeout=800):
                button.click(timeout=1500)
                polite_sleep(0.5, 1.0)
        except Exception:
            pass


def open_reviews_section(page):
    """Scroll down in steps so lazily loaded review content appears, and click a 'Reviews' tab if there is one."""
    for _ in range(8):
        page.mouse.wheel(0, 1400)
        time.sleep(0.6)
    try:
        tab = page.get_by_role("tab", name=re.compile("review", re.I)).first
        if tab.is_visible(timeout=1000):
            tab.click(timeout=2000)
    except Exception:
        pass


def select_sort(page, label):
    """Choose a review sort order by its visible name. Returns True if it worked, False if the option was not found."""
    try:
        for select in page.locator("select").all():
            options = [o.strip() for o in select.locator("option").all_inner_texts()]
            match = next((o for o in options if o.lower() == label.lower()), None)
            if match:
                select.select_option(label=match)
                polite_sleep()
                return True
        trigger = page.get_by_role("button", name=re.compile("sort", re.I)).first
        trigger.click(timeout=2000)
        page.get_by_role("option", name=re.compile(re.escape(label), re.I)).first.click(timeout=2000)
        polite_sleep()
        return True
    except Exception:
        return False


def load_all_reviews(page, max_clicks):
    """Click 'Show more' until the button disappears, the review count stops growing, or the click limit is reached."""
    clicks, stale = 0, 0
    last_size = len(page.content())
    while clicks < max_clicks and stale < 3:
        button = page.get_by_role("button", name=re.compile(r"show more|load more|see more reviews", re.I)).first
        try:
            visible = button.is_visible(timeout=1500)
        except Exception:
            visible = False
        if not visible:
            break
        check_not_blocked(page)
        button.scroll_into_view_if_needed()
        button.click(timeout=4000)
        clicks += 1
        polite_sleep()
        size = len(page.content())
        stale = stale + 1 if size == last_size else 0
        last_size = size
    return clicks


def scrape_reviews(url, sort_labels=None, headless=None):
    """Collect reviews for one product page across all sort options. Returns a list of review dictionaries."""
    from playwright.sync_api import sync_playwright

    sort_labels = sort_labels or SORT_LABELS
    headless = HEADLESS if headless is None else headless
    collected, report = {}, []
    with sync_playwright() as playwright:
        launch_args = {"headless": headless}
        if PROXY_LIST:
            launch_args["proxy"] = {"server": random.choice(PROXY_LIST)}
        browser = playwright.chromium.launch(**launch_args)
        context = browser.new_context(user_agent=random.choice(USER_AGENTS), locale="en-CA", viewport={"width": 1366, "height": 900})
        page = context.new_page()
        try:
            with_retries(lambda: page.goto(url, wait_until="domcontentloaded", timeout=45000), "open page")
            polite_sleep(2, 4)
            check_not_blocked(page)
            dismiss_popups(page)
            open_reviews_section(page)
            for label in sort_labels:
                if not select_sort(page, label):
                    report.append((label, "sort option not found, skipped", 0, 0))
                    continue
                clicks = load_all_reviews(page, MAX_SHOW_MORE_CLICKS)
                found = extract_reviews(page.content())
                for review in found:
                    key = review_key(review)
                    entry = collected.setdefault(key, {**review, "found_via": []})
                    entry["found_via"].append(label)
                report.append((label, "ok", clicks, len(found)))
        except BlockedError as error:
            print("STOPPED:", error)
        finally:
            context.close()
            browser.close()
    print(pd.DataFrame(report, columns=["sort option", "result", "show-more clicks", "reviews on page"]).to_string(index=False))
    return [{"key": k, **v} for k, v in collected.items()]

### 4.1 Inspect cell (use this if the scraper finds zero reviews)
It opens the page, saves its HTML to `data/page_snapshot.html`, and prints what it can see: how many embedded review objects, which buttons and sort dropdowns exist, and how many elements have review-like class names. Use that to set `SELECTORS` in section 2.

In [7]:
def inspect_page(url):
    """Open the page once and print what is on it, so selectors can be chosen from facts instead of guesses."""
    from playwright.sync_api import sync_playwright

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=HEADLESS)
        page = browser.new_context(user_agent=USER_AGENTS[0], locale="en-CA").new_page()
        page.goto(url, wait_until="domcontentloaded", timeout=45000)
        polite_sleep(2, 4)
        dismiss_popups(page)
        open_reviews_section(page)
        html = page.content()
        Path("data/page_snapshot.html").write_text(html, encoding="utf-8")
        print("embedded Review objects:", len(reviews_from_jsonld(html)))
        print("buttons mentioning more:", [b.strip() for b in page.locator("button").all_inner_texts() if re.search("more", b, re.I)][:10])
        print("dropdowns and their options:", [s.locator("option").all_inner_texts() for s in page.locator("select").all()][:5])
        soup = BeautifulSoup(html, "html.parser")
        classes = [" ".join(t.get("class", [])) for t in soup.find_all(class_=re.compile("review", re.I))]
        print("elements with 'review' in the class name:", len(classes))
        print("sample class names:", sorted(set(classes))[:15])
        browser.close()

In [8]:
RUN_SCRAPER = False

if RUN_SCRAPER and PRODUCT_URL.startswith("http"):
    raw_reviews = scrape_reviews(PRODUCT_URL)
    print("unique reviews collected:", len(raw_reviews))
    pd.DataFrame(raw_reviews).to_json("data/raw_reviews.json", orient="records", indent=1)
else:
    print("Set RUN_SCRAPER = True (and PRODUCT_URL) to run. Use inspect_page(PRODUCT_URL) first if you are unsure.")

Set RUN_SCRAPER = True (and PRODUCT_URL) to run. Use inspect_page(PRODUCT_URL) first if you are unsure.


## 5. Build the dataset
We turn the collected reviews into the required table: primary key, title, review text, date (`YYYY-MM-DD`), star rating out of 5, source, reviewer name. We also keep which sort options found each review.

Cleaning rules: collapse whitespace, drop reviews with no text at all, drop exact duplicates (same key), and report how many reviews had a missing date or rating instead of silently filling them.

In [9]:
def build_dataset(raw_reviews):
    """Turn raw review dictionaries into the final table with the required columns and tidy values."""
    frame = pd.DataFrame(raw_reviews)
    frame["title"] = frame["title"].map(clean_space)
    frame["body"] = frame["body"].map(clean_space)
    frame = frame[(frame["title"] != "") | (frame["body"] != "")].drop_duplicates("key")
    table = pd.DataFrame({
        "Primary Key": frame["key"],
        "Title": frame["title"],
        "Review Text": frame["body"],
        "Date": frame["date"],
        "Rating": frame["rating"],
        "Source": SOURCE_NAME,
        "Reviewer Name": frame["author"].replace("", "Anonymous"),
        "Found Via Sort": frame["found_via"].map(lambda v: ", ".join(sorted(set(v)))),
    }).reset_index(drop=True)
    print("reviews:", len(table), "| missing date:", int(table["Date"].isna().sum()), "| missing rating:", int(table["Rating"].isna().sum()))
    return table


if Path("data/raw_reviews.json").exists():
    reviews = build_dataset(json.loads(Path("data/raw_reviews.json").read_text()))
    reviews.head()
else:
    reviews = None
    print("No scraped data yet. Run section 4 first.")

No scraped data yet. Run section 4 first.


## 6. Sentiment analysis
**Method.** VADER is a lexicon-based sentiment tool built for short, informal text such as reviews. It handles negation ("not good"), intensifiers ("very good") and exclamation marks, needs no training and no GPU, and is easy to explain.

**Steps.**
1. *Preprocess:* join title and body, remove web addresses, collapse spaces. Punctuation and capital letters are kept because VADER uses them.
2. *Score the text* with VADER to get a compound score from -1 to +1.
3. *Combine with the star rating.* Text alone can be wrong (sarcasm, short reviews) and the rating is the customer's own summary, so the final score is the average of the text score and the rating converted to the same scale: (rating - 3) / 2.
4. *Label:* score at or above +0.15 is Positive, at or below -0.15 is Negative, otherwise Neutral. If a review has no rating, the text score alone is used.

**Optional upgrade:** a transformer model usually handles sarcasm better. See the optional cell at the end of this section.

In [10]:
analyzer = SentimentIntensityAnalyzer()
POS_CUTOFF = 0.15
NEG_CUTOFF = -0.15


def preprocess(title, body):
    """Join the title and body into one text, remove web addresses and collapse spaces."""
    text = clean_space(f"{title}. {body}" if title and body else (title or body))
    return re.sub(r"https?://\S+", " ", text).strip()


def sentiment_scores(text, rating):
    """Return (text score, blended score, label) for one review."""
    text_score = analyzer.polarity_scores(text)["compound"]
    if rating is None or pd.isna(rating):
        blended = text_score
    else:
        blended = (text_score + (float(rating) - 3) / 2) / 2
    label = "Positive" if blended >= POS_CUTOFF else "Negative" if blended <= NEG_CUTOFF else "Neutral"
    return text_score, blended, label


def add_sentiment(table):
    """Add clean text, the three sentiment columns and return the new table."""
    table = table.copy()
    table["Clean Text"] = [preprocess(t, b) for t, b in zip(table["Title"], table["Review Text"])]
    scored = [sentiment_scores(t, r) for t, r in zip(table["Clean Text"], table["Rating"])]
    table["Text Score"] = [round(s[0], 3) for s in scored]
    table["Sentiment Score"] = [round(s[1], 3) for s in scored]
    table["Sentiment Label"] = [s[2] for s in scored]
    return table


tests = [("Love it", "Works perfectly, very happy with the quality!", 5), ("Terrible", "Stopped working after a week. Not recommended.", 1), ("Okay", "It is fine, nothing special.", 3)]
for title, body, stars in tests:
    print(stars, "stars ->", sentiment_scores(preprocess(title, body), stars)[2])
print("(these three lines are a self-test on invented text)")

5 stars -> Positive
1 stars -> Negative
3 stars -> Neutral
(these three lines are a self-test on invented text)


### 6.1 Optional: transformer model
Slower but better with sarcasm and context. It downloads a model of about 500 MB the first time. Leave `USE_TRANSFORMER = False` unless you want to compare.

In [11]:
USE_TRANSFORMER = False

if USE_TRANSFORMER and reviews is not None:
    from transformers import pipeline

    classifier = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", truncation=True, max_length=256)
    reviews["Transformer Label"] = [classifier(t)[0]["label"].title() for t in [preprocess(a, b) for a, b in zip(reviews["Title"], reviews["Review Text"])]]

## 7. Aspect tags
The assignment asks for tags such as `['Good Design & Quality (Pos)']` or `['Satisfactory & Recommended (Pos)']`: **what the review talks about, plus how the customer feels about it.**

**Method.** A small dictionary lists words for each aspect. A review is split into sentences. For every sentence that mentions an aspect, we score that sentence alone with VADER and attach (Pos), (Neg) or (Neu). This lets one review say "great quality (Pos)" and "poor battery (Neg)" together.

The dictionary below is generic so it works for any product. Edit the word lists after reading some of your own reviews: adding the words people really use for your product is the single biggest quality gain.

In [12]:
ASPECTS = {
    "Design & Quality": {"pos": "Good Design & Quality", "neg": "Poor Design & Quality",
                         "words": r"quality|design|look|looks|sturdy|solid|well made|well-made|built|build|material|finish|sleek|color|colour|beautiful|pretty|stylish|cheap|flimsy"},
    "Performance & Function": {"pos": "Works Well", "neg": "Performance Problems",
                               "words": r"work|works|working|performance|perform|fast|slow|function|feature|features|lag|freez|crash|connect|signal|sound|picture|screen|display|power"},
    "Ease of Use & Setup": {"pos": "Easy to Use & Set Up", "neg": "Hard to Use or Set Up",
                            "words": r"easy|simple|setup|set up|install|assemble|assembly|instructions|intuitive|user.?friendly|difficult|complicated|confusing"},
    "Value for Money": {"pos": "Good Value", "neg": "Poor Value",
                        "words": r"price|value|worth|money|cost|expensive|affordable|cheap|deal|bargain|overpriced"},
    "Durability & Reliability": {"pos": "Durable & Reliable", "neg": "Durability Problems",
                                 "words": r"durable|last|lasted|lasting|broke|broken|break|defect|defective|faulty|stopped|died|failure|failed|reliable|return|returned|replace|replacement"},
    "Comfort & Fit": {"pos": "Comfortable & Good Fit", "neg": "Uncomfortable or Poor Fit",
                      "words": r"comfort|comfortable|fit|fits|size|sizing|snug|tight|loose|heavy|light|lightweight|bulky|noisy|quiet"},
    "Battery & Power": {"pos": "Good Battery Life", "neg": "Battery or Power Issues",
                        "words": r"battery|charge|charging|charger|power|plug|cord|cable"},
    "Customer Service & Delivery": {"pos": "Good Service & Delivery", "neg": "Poor Service or Delivery",
                                    "words": r"service|support|staff|delivery|delivered|shipping|shipped|arrived|packaging|package|warranty|refund|store"},
}
RECOMMEND_WORDS = r"recommend|satisfied|happy|love|loved|great|excellent|perfect|awesome|good|nice|best|pleased|impressed"
DISAPPOINT_WORDS = r"disappoint|regret|waste|worst|terrible|awful|horrible|useless|not recommend|do not buy|don't buy|never again"


def sentence_split(text):
    """Split text into sentences at full stops, question marks, exclamation marks and line breaks."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n", text) if s.strip()]


def tag_to_label(score):
    """Convert a sentence sentiment score into Pos, Neg or Neu."""
    return "Pos" if score >= 0.05 else "Neg" if score <= -0.05 else "Neu"


def aspect_tags(text, overall_label):
    """Return the list of aspect tags for one review, for example ['Good Design & Quality (Pos)'].

    Only sentences that mention an aspect are scored. If no aspect is mentioned, a general tag is built from the overall
    sentiment so that every review has at least one tag.
    """
    tags = []
    for sentence in sentence_split(text):
        lowered = sentence.lower()
        score = analyzer.polarity_scores(sentence)["compound"]
        for aspect in ASPECTS.values():
            if re.search(r"\b(?:" + aspect["words"] + r")", lowered):
                label = tag_to_label(score)
                if label == "Neu":
                    continue
                name = aspect["pos"] if label == "Pos" else aspect["neg"]
                tag = f"{name} ({label})"
                if tag not in tags:
                    tags.append(tag)
    if not tags:
        lowered = text.lower()
        if overall_label == "Positive":
            tags = ["Satisfactory & Recommended (Pos)"] if re.search(RECOMMEND_WORDS, lowered) else ["General Satisfaction (Pos)"]
        elif overall_label == "Negative":
            tags = ["Disappointing & Not Recommended (Neg)"] if re.search(DISAPPOINT_WORDS, lowered) else ["General Dissatisfaction (Neg)"]
        else:
            tags = ["General Comment (Neu)"]
    return tags


test_text = "Very nice! Colour is very pretty and easy to assemble. Battery died after two days though. Bought for my granddaughter."
print(aspect_tags(test_text, "Positive"))
print("(self-test on invented text)")

['Good Design & Quality (Pos)', 'Easy to Use & Set Up (Pos)', 'Durability Problems (Neg)', 'Battery or Power Issues (Neg)']
(self-test on invented text)


## 8. Run sentiment and tagging on the scraped reviews

In [13]:
if reviews is not None:
    reviews = add_sentiment(reviews)
    reviews["Sentiment"] = [str(aspect_tags(t, l)) for t, l in zip(reviews["Clean Text"], reviews["Sentiment Label"])]
    reviews["Aspect Tags"] = [aspect_tags(t, l) for t, l in zip(reviews["Clean Text"], reviews["Sentiment Label"])]
    print(reviews["Sentiment Label"].value_counts().to_string())
    reviews[["Primary Key", "Title", "Review Text", "Rating", "Sentiment Label", "Sentiment"]].head(8)

## 9. Analysis for stakeholders
Four questions: how happy are customers overall, do text sentiment and star rating agree, what do customers like most, and what do they complain about most. Counts count each review once per tag.

In [14]:
if reviews is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    reviews["Rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title("Star ratings")
    axes[0].set_xlabel("stars")
    reviews["Sentiment Label"].value_counts().reindex(["Positive", "Neutral", "Negative"]).plot(kind="bar", ax=axes[1], color=["seagreen", "gray", "firebrick"])
    axes[1].set_title("Sentiment label")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "sentiment_overview.png", dpi=150)
    plt.show()
    print(pd.crosstab(reviews["Rating"], reviews["Sentiment Label"]))

In [15]:
if reviews is not None:
    tag_counts = reviews["Aspect Tags"].explode().value_counts()
    positives = tag_counts[tag_counts.index.str.endswith("(Pos)")].head(8)
    negatives = tag_counts[tag_counts.index.str.endswith("(Neg)")].head(8)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    positives[::-1].plot(kind="barh", ax=axes[0], color="seagreen")
    axes[0].set_title("What customers like (top tags)")
    negatives[::-1].plot(kind="barh", ax=axes[1], color="firebrick")
    axes[1].set_title("What customers dislike (top tags)")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "aspect_drivers.png", dpi=150)
    plt.show()

    def examples(tag, n=2):
        """Return up to n short example review snippets that carry a given tag."""
        rows = reviews[reviews["Aspect Tags"].map(lambda t: tag in t)].head(n)
        return [clean_space(f"{a} {b}")[:160] for a, b in zip(rows["Title"], rows["Review Text"])]

    lines = ["**Insights summary (auto-generated from the scraped reviews)**", ""]
    lines.append(f"Reviews analysed: {len(reviews)}. Average rating: {reviews['Rating'].mean():.2f} out of 5.")
    shares = reviews["Sentiment Label"].value_counts(normalize=True).mul(100).round(0)
    lines.append("Sentiment mix: " + ", ".join(f"{k} {int(v)}%" for k, v in shares.items()) + ".")
    lines += ["", "**Top satisfaction drivers**"]
    for tag, n in positives.head(5).items():
        sample = examples(tag, 1)
        lines.append(f"- {tag}: {n} reviews. Example: " + (sample[0] if sample else "none"))
    lines += ["", "**Top dissatisfaction pain points**"]
    for tag, n in negatives.head(5).items():
        sample = examples(tag, 1)
        lines.append(f"- {tag}: {n} reviews. Example: " + (sample[0] if sample else "none"))
    (OUT_DIR / "insights_summary.md").write_text("\n".join(lines), encoding="utf-8")
    print("\n".join(lines))

## 10. Export
The final file has the required scraped fields plus the sentiment label, the score and the aspect tags in the same style as the assignment's example.

In [16]:
if reviews is not None:
    final = reviews[["Primary Key", "Title", "Review Text", "Date", "Rating", "Source", "Reviewer Name",
                     "Found Via Sort", "Sentiment Label", "Sentiment Score", "Sentiment"]]
    final.to_csv(OUT_DIR / "task1_reviews_with_sentiment.csv", index=False)
    final.to_excel(OUT_DIR / "task1_reviews_with_sentiment.xlsx", index=False)
    print("saved", len(final), "reviews to", OUT_DIR)

## 11. What to write in the report
Open `outputs/insights_summary.md` and the two charts in `outputs`. The report template (`Task1_Report_TEMPLATE.docx`) already contains the anti-scraping section. Fill in the insights section with: the sentiment mix, the top 3 to 5 satisfaction drivers, the top 3 to 5 pain points with an example quote each, and 3 to 4 recommendations that each point back to a specific tag count.

**Limits to state honestly:** lexicon sentiment can miss sarcasm; aspect tags come from keyword lists; relative dates are approximate; only reviews visible to the public on the chosen product were collected.